In [1]:
from google.colab import drive

In [2]:
RUN_FROM_COLAB = True

In [3]:
import sklearn
print(sklearn.__version__)

1.6.1


In [4]:
import xgboost
print(xgboost.__version__)

3.4.1


In [5]:
import joblib
print(joblib.__version__)

1.5.3


In [6]:
if RUN_FROM_COLAB:
  drive.mount('/content/drive')
  DATA_PATH = '/content/drive/MyDrive/sms-spam-classification-fastapi/data'
  MODELS_PATH = '/content/drive/MyDrive/sms-spam-classification-fastapi/models'
else:
  DATA_PATH = '/sms-spam-classification-fastapi/data/spam.csv'
  MODELS_PATH = '/sms-spam-classification-fastapi/models'


Mounted at /content/drive


In [7]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
df = pd.read_csv(DATA_PATH+'/spam.csv', usecols=['v1', 'v2'], encoding='ISO-8859-1').rename(columns={'v1':'label', 'v2':'text'})
df

,label,text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
5567,spam,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate."
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other suggestions?"
5570,ham,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [9]:
from sklearn.preprocessing import LabelEncoder

In [10]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
df

,label,text
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
5567,1,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate."
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other suggestions?"
5570,0,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free


In [11]:
df.groupby('label').count()

,text
label,
0,4825
1,747


In [12]:
import spacy
!python -m spacy download en_core_web_md
nlp = spacy.load('en_core_web_md')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import re
import os
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    doc = nlp(text)
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    text = re.sub(r'\b\w\b', ' ', text) # single characters

    return text

if not os.path.isfile(DATA_PATH+'/normalized_text.pkl'):
    df['text'] = df['text'].apply(normalize_text)
    pd.to_pickle(df['text'], DATA_PATH+'/normalized_text.pkl')
else:
    df['text'] = pd.read_pickle(DATA_PATH+'/normalized_text.pkl')
df

,label,text
0,0,jurong point crazy available bugis great world la buffet cine get amore wat
1,0,ok lar joke wif oni
2,1,free entry wkly comp win fa cup final tkts st text fa receive entry question std txt rate apply
3,0,dun early hor
4,0,nah don think go usf live
...,...,...
5567,1,nd time try contact win pound prize claim easy minute bt national rate
5568,0,go esplanade fr home
5569,0,pity mood suggestion
5570,0,guy bitch act like interested buy week give free


In [14]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy import sparse
import joblib

In [15]:
if not os.path.isfile(DATA_PATH+'/BoW.npz'):
    cv = CountVectorizer(ngram_range=(1, 2))
    bag_of_words = cv.fit_transform(df['text'])
    joblib.dump(cv, MODELS_PATH+'/count_vectorizer.joblib')
    sparse.save_npz(DATA_PATH+'/BoW.npz', bag_of_words)
else:
    cv = joblib.load(MODELS_PATH+'/count_vectorizer.joblib')
    bag_of_words = sparse.load_npz(DATA_PATH+'/BoW.npz')
bag_of_words

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 77486 stored elements and shape (5572, 32527)>

In [16]:
if not os.path.isfile(DATA_PATH+'/tfidf.npz'):
    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    tfidf = tfidf_vectorizer.fit_transform(df['text'])
    joblib.dump(tfidf_vectorizer, MODELS_PATH+'/tfidf_vectorizer.joblib')
    sparse.save_npz(DATA_PATH+'/tfidf.npz', tfidf)
else:
    tfidf_vectorizer = joblib.load(MODELS_PATH+'/tfidf_vectorizer.joblib')
    tfidf = sparse.load_npz(DATA_PATH+'/tfidf.npz')
tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 77486 stored elements and shape (5572, 32527)>

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline

In [18]:
X_train, X_test, y_train, y_test = train_test_split(bag_of_words, df['label'], test_size=0.25, random_state=42)

In [19]:
if not os.path.isfile(MODELS_PATH+'/nb_pipeline.joblib'):
    gcv = GridSearchCV(estimator=MultinomialNB(), param_grid={'alpha': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 2.0], 'fit_prior': [True, False]}, cv=5, scoring='f1')
    gcv.fit(X_train, y_train)
    nb = gcv.best_estimator_
    nb_pipeline = make_pipeline(cv, nb)
    joblib.dump(nb_pipeline, MODELS_PATH+'/nb_pipeline.joblib')
else:
    nb_pipeline = joblib.load(MODELS_PATH+'/nb_pipeline.joblib')

y_pred = nb_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1202
           1       0.89      0.93      0.91       191

    accuracy                           0.97      1393
   macro avg       0.94      0.95      0.94      1393
weighted avg       0.97      0.97      0.97      1393



In [20]:
from sklearn.linear_model import LogisticRegression

In [21]:
X_train, X_test, y_train, y_test = train_test_split(tfidf, df['label'], test_size=0.25, random_state=42)

In [22]:
if not os.path.isfile(MODELS_PATH+'/logreg_pipeline.joblib'):
    gcv = GridSearchCV(estimator=LogisticRegression(), param_grid={'penalty':['l2'], 'C':[1.0, 0.1, 0.5, 10] ,'solver':['lbfgs', ], 'max_iter':[100, 500, 1000]}, cv=5, scoring='f1')
    gcv.fit(X_train, y_train)
    logreg = gcv.best_estimator_
    logreg_pipeline = make_pipeline(tfidf_vectorizer, logreg)
    joblib.dump(logreg_pipeline, MODELS_PATH+'/logreg_pipeline.joblib')
else:
    logreg_pipeline = joblib.load(MODELS_PATH+'/logreg_pipeline.joblib')

y_pred = logreg_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1202
           1       0.98      0.77      0.87       191

    accuracy                           0.97      1393
   macro avg       0.97      0.89      0.92      1393
weighted avg       0.97      0.97      0.97      1393



In [23]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [24]:
param_distributions = {
    "max_depth": [3, 5, 7, 9],
    "min_child_weight": [1, 3, 5, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "n_estimators": [200, 500, 1000],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.5, 1, 5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10],
}
if not os.path.isfile(MODELS_PATH+'/xgboost_pipeline.joblib'):
    rscv = RandomizedSearchCV(estimator=XGBClassifier(n_jobs=-1), param_distributions=param_distributions, random_state=42, cv=5, scoring='f1', n_iter=20)
    rscv.fit(X_train, y_train)
    xgbst = rscv.best_estimator_
    xgboost_pipeline = make_pipeline(tfidf_vectorizer, xgbst)
    joblib.dump(xgboost_pipeline, MODELS_PATH+'/xgboost_pipeline.joblib')
else:
    xgboost_pipeline = joblib.load(MODELS_PATH+'/xgboost_pipeline.joblib')

y_pred = xgboost_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1202
           1       0.98      0.82      0.89       191

    accuracy                           0.97      1393
   macro avg       0.98      0.91      0.94      1393
weighted avg       0.97      0.97      0.97      1393



In [25]:
from sklearn.ensemble import VotingClassifier

In [26]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.25, random_state=42)

In [27]:
X_train

,text
4281,
585,tell headache want use hour sick time
4545,try weight tear come ur heart fall ur eye remember stupid friend share bslvyl
3034,raji pls favour pls convey birthday wish nimya pls today birthday
2758,time prob
...,...
3772,come hostel go sleep plz class hrishi
5191,sorry ll later
5226,prabha soryda realy frm heart sory
5390,not joke seriously told


In [28]:
if not os.path.isfile(MODELS_PATH+'/voting_classifier.joblib'):
    vc = VotingClassifier(estimators=[('naive_bayes', nb_pipeline), ('logistic_regression', logreg_pipeline), ('xgboost', xgboost_pipeline)], voting='soft')
    gscv = GridSearchCV(estimator=vc, param_grid={'weights': [[1, 1, 2], [1, 1, 3], [1, 2, 1], [1, 2, 2]]}, cv=5, scoring='f1')
    gscv.fit(X_train, y_train)
    voting_classifier = gscv.best_estimator_
    joblib.dump(voting_classifier, MODELS_PATH+'/voting_classifier.joblib')
else:
  voting_classifier = joblib.load(MODELS_PATH+'/voting_classifier.joblib')

y_pred = voting_classifier.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1202
           1       0.98      0.87      0.92       191

    accuracy                           0.98      1393
   macro avg       0.98      0.94      0.96      1393
weighted avg       0.98      0.98      0.98      1393



In [29]:
voting_classifier.weights

[1, 2, 1]